# ads-bib Colab quickstart: Gemma preset

This notebook runs the full `ads-bib` pipeline in Google Colab with a T4 GPU and the unchanged `local_gpu` preset.

Before running: choose `Runtime > Change runtime type > T4 GPU`.


## 1. Install

This installs the package and the matching CUDA Torch/TorchVision runtime used by the notebook.


In [ ]:
%pip install -q uv
!uv pip install --system -q --extra-index-url https://download.pytorch.org/whl/cu124 \
    ads-bib "torch==2.6.0" "torchvision==0.21.0" bitsandbytes
!uv pip uninstall --system -q torchcodec || true


## 2. Check the GPU

If this cell fails, switch the Colab runtime to T4 GPU and run the notebook again.


In [ ]:
import logging
import warnings

import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU found. In Colab choose Runtime > Change runtime type > T4 GPU, "
        "then run this notebook again."
    )

print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

warnings.filterwarnings("ignore")
logging.getLogger("torchao").setLevel(logging.ERROR)
transformers.logging.set_verbosity_error()


## 3. Add tokens

Create an ADS token at [ADS token settings](https://ui.adsabs.harvard.edu/user/settings/token). This Gemma notebook also needs a Hugging Face token from [HF token settings](https://huggingface.co/settings/tokens) after you have access to the preset models: [TranslateGemma](https://huggingface.co/google/translategemma-4b-it), [EmbeddingGemma](https://huggingface.co/google/embeddinggemma-300m), and [Gemma 3 1B](https://huggingface.co/google/gemma-3-1b-it).

`/content/ads-bib-colab` is temporary. Download anything you want to keep before the Colab session ends.


In [ ]:
from getpass import getpass
import os
from pathlib import Path

PROJECT_ROOT = "/content/ads-bib-colab"
Path(PROJECT_ROOT).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_ROOT)

ads_token = None
hf_token = None
try:
    from google.colab import userdata

    ads_token = userdata.get("ADS_TOKEN") or userdata.get("ADS_API_TOKEN")
    hf_token = userdata.get("HF_TOKEN") or userdata.get("HUGGINGFACE_TOKEN")
except Exception:
    pass

if not ads_token:
    ads_token = getpass("ADS_TOKEN (https://ui.adsabs.harvard.edu/user/settings/token): ")
if not ads_token:
    raise ValueError("ADS_TOKEN is required.")
os.environ["ADS_TOKEN"] = ads_token
print("ADS token loaded.")

if not hf_token:
    hf_token = getpass("HF_TOKEN for Gemma model access: ") or None

if not hf_token:
    raise ValueError("HF_TOKEN is required for the Gemma preset notebook.")

os.environ["HF_TOKEN"] = hf_token
print("HF token loaded.")
print(f"Project folder: {PROJECT_ROOT}")


## 4. Choose the ADS search

This uses the official `local_gpu` preset as-is.


In [ ]:
from ads_bib.presets import preset_to_dict

SEARCH_QUERY = 'author:"Hawking, S*"'
RUN_NAME = "ads_bib_colab_hawking_gemma"

CONFIG = preset_to_dict("local_gpu")
CONFIG["run"]["run_name"] = RUN_NAME
CONFIG["run"]["project_root"] = PROJECT_ROOT
CONFIG["search"]["query"] = SEARCH_QUERY
CONFIG["author_disambiguation"]["enabled"] = True

print("Search query:", SEARCH_QUERY)
print("Translation:", CONFIG["translate"]["model"])
print("Embeddings:", CONFIG["topic_model"]["embedding_model"])
print("Topic labels:", CONFIG["topic_model"]["llm_model"])
print("Author disambiguation:", "enabled")


## 5. Prepare models

Download and warm the Gemma preset models before the pipeline starts. The run cell stays focused on pipeline stages.


In [ ]:
from ads_bib.bootstrap import ensure_default_fasttext_model
from ads_bib.translate import _load_local_transformers_translation_model
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import gc
import torch

print("Preparing fastText language model...")
ensure_default_fasttext_model(
    project_root=PROJECT_ROOT,
    configured_path=CONFIG["translate"]["fasttext_model"],
)

print("Preparing translation model...")
translation_stack = _load_local_transformers_translation_model(
    CONFIG["translate"]["model"],
    runtime_log_path=None,
)
del translation_stack

print("Preparing embedding model...")
embedding_model = SentenceTransformer(CONFIG["topic_model"]["embedding_model"])
del embedding_model

print("Preparing BERTopic keyword model...")
keybert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
del keybert_model

print("Preparing topic label model...")
label_model_id = CONFIG["topic_model"]["llm_model"]
label_tokenizer = AutoTokenizer.from_pretrained(label_model_id)
try:
    label_model = AutoModelForCausalLM.from_pretrained(
        label_model_id,
        device_map="auto",
        dtype="auto",
    )
except TypeError:
    label_model = AutoModelForCausalLM.from_pretrained(
        label_model_id,
        device_map="auto",
        torch_dtype="auto",
    )
del label_model, label_tokenizer
gc.collect()
torch.cuda.empty_cache()

print("Models are ready.")


## 6. Run the pipeline

This runs search, export, translation, tokenization, author disambiguation, embeddings, topic modeling, visualization, curation, and citations.


In [ ]:
from ads_bib.runner import load_run_config, run_resolved_config

resolved_config = load_run_config(
    config=CONFIG,
    project_root=PROJECT_ROOT,
)

result = run_resolved_config(
    resolved_config,
    run_name=RUN_NAME,
    project_root=PROJECT_ROOT,
    output_mode="notebook",
)

print("Run finished.")
print(f"Run folder: {result.run.paths['root']}")


## 7. Results

The most useful outputs are the run folder, the topic map, the ADS export files, and the citation files.


In [ ]:
cols = [c for c in ["Bibcode", "Year", "Author", "Title"] if c in result.publications.columns]

display(result.publications[cols].head(10))
display(result.topic_info.head(20))


In [ ]:
run_root = result.run.paths["root"]
topic_map = result.run.paths["plots"] / "topic_map.html"
export_dir = result.run.paths["export"]
citations_dir = result.run.paths["citations"]

print(f"Run folder: {run_root}")
print(f"Export files: {export_dir}")
print(f"Citation files: {citations_dir}")
print(f"Topic map: {topic_map}")

topic_map_obj = getattr(result, "topic_map", None)
if topic_map_obj is None and result.topic_df is not None:
    from ads_bib.visualize import create_topic_map

    vis_config = CONFIG.get("visualization", {})
    topic_map_obj = create_topic_map(
        result.topic_df,
        title=vis_config.get("title", "ADS Topic Map"),
        subtitle=vis_config.get("subtitle", ""),
        dark_mode=vis_config.get("dark_mode", True),
        font_family=vis_config.get("font_family", "Cinzel"),
        topic_tree=vis_config.get("topic_tree", False),
    )

topic_map_obj
